In [ ]:
from astropy.io import fits
from astropy.coordinates import SkyCoord
from astropy.coordinates import ICRS, Galactic, FK4, FK5  # Low-level frames
import astropy.units as u
import numpy as np
import matplotlib.pyplot as plt
from reproject.mosaicking import find_optimal_celestial_wcs
from reproject import reproject_interp
from reproject.mosaicking import reproject_and_coadd
from astropy.wcs import WCS
from astropy.utils.data import get_pkg_data_filename
from astropy.convolution import Gaussian2DKernel
#from scipy.signal import convolve as scipy_convolve
from astropy.convolution import convolve
import gc
from mpl_toolkits.axes_grid1 import make_axes_locatable

In [ ]:
#dir_in = '/home2/DATA_AO/CGPS_GMIMS'
dir_in = '/home/aordog/DATA/CGPS_GMIMS/'

## Read in RM files:

In [ ]:
hdu_CG_RM = fits.open(dir_in+'RM_CG_conv2.fits')
CG_RM = hdu_CG_RM[0].data
print(CG_RM.shape)

hdu_G_RM = fits.open(dir_in+'RM_G.fits')
G_RM = hdu_G_RM[0].data
print(G_RM.shape)

hdu_C_RM = fits.open(dir_in+'RM_C_conv2.fits')
C_RM = hdu_C_RM[0].data
print(C_RM.shape)
  

## Read in PI files:

In [ ]:
hdu_CG_PI = fits.open(dir_in+'PI_CG_conv2.fits')
CG_PI = hdu_CG_PI[0].data
print(CG_PI.shape)

hdu_G_PI = fits.open(dir_in+'PI_G.fits')
G_PI = hdu_G_PI[0].data
print(G_PI.shape)

hdu_C_PI = fits.open(dir_in+'PI_C_conv2.fits')
C_PI = hdu_C_PI[0].data
print(C_PI.shape)  

## Individual band PI:

In [ ]:
band_PI = False

if band_PI:

    CG_PI_list = []
    G_PI_list = []
    C_PI_list = []

    band = ['A','B','C','D']
    bandlc = ['a','b','c','d']
    for i in range(0,4):

        hdu_CG_PI = fits.open(dir_in+'PI'+band[i]+'_CG.fits')
        hdu_G_PI = fits.open(dir_in+'PI'+band[i]+'_G.fits')
        hdu_C_PI = fits.open(dir_in+'PI'+band[i]+'_C.fits')
        CG_PI_list.append(hdu_CG_PI[0].data)
        G_PI_list.append(hdu_G_PI[0].data)
        C_PI_list.append(hdu_C_PI[0].data)

In [ ]:
#G_PI = np.nanmean(np.array(G_PI_list),axis=0)
#C_PI = np.nanmean(np.array(C_PI_list),axis=0)
#CG_PI = np.nanmean(np.array(CG_PI_list),axis=0)

In [ ]:
#del CG_PI_list, G_PI_list,C_PI_list
#gc.collect()

## Set PI threshold:

In [ ]:
PI_lim = 0.1

wGbad = np.where(G_PI < PI_lim)
wCGbad = np.where(CG_PI < PI_lim)

CG_RM_good = CG_RM.copy()
CG_RM_good[wCGbad] = np.nan

C_RM_good = C_RM.copy()
C_RM_good[wCGbad] = np.nan

#G_RM_good = G_RM.copy()
#G_RM_good[wGbad] = np.nan

del wGbad,wCGbad
gc.collect()

## Plot all PI and RM

In [ ]:
fs = 18
fig = plt.figure(figsize=(20, 16))

PIlim = 0.4
RMlim = 400

c = SkyCoord([100,95], [-1,3], frame=Galactic, unit="deg")
wcs = WCS(hdu_G_PI[0].header)
x, y = np.round(wcs.world_to_pixel(c))
i1 = int(x[0])
i2 = int(x[1])
j1 = int(y[0])
j2 = int(y[1])

plt.subplot(331,projection=wcs[j1:j2,i1:i2])
plt.imshow(G_PI[j1:j2,i1:i2], vmin=0, vmax=PIlim, origin='lower')
plt.contour(G_PI[j1:j2,i1:i2], levels=[PI_lim], colors='white', alpha=0.5)
plt.xlabel('Galactic Longitude')
plt.ylabel('Galactic Latitude')

plt.subplot(332,projection=wcs[j1:j2,i1:i2])
plt.imshow(C_PI[j1:j2,i1:i2], vmin=0, vmax=PIlim, origin='lower')
plt.contour(G_PI[j1:j2,i1:i2], levels=[PI_lim], colors='white', alpha=0.5)
plt.xlabel('Galactic Longitude')
plt.ylabel('Galactic Latitude')

plt.subplot(333,projection=wcs[j1:j2,i1:i2])
plt.imshow(CG_PI[j1:j2,i1:i2], vmin=0, vmax=PIlim, origin='lower')
plt.contour(G_PI[j1:j2,i1:i2], levels=[PI_lim], colors='white', alpha=0.5)
plt.xlabel('Galactic Longitude')
plt.ylabel('Galactic Latitude')


plt.subplot(334,projection=wcs[j1:j2,i1:i2])
#plt.imshow(G_RM[j1:j2,i1:i2],vmin=-RMlim,vmax=RMlim,origin='lower',cmap='RdBu')
plt.contour(G_PI[j1:j2,i1:i2], levels=[PI_lim], colors='k', alpha=0.5)
plt.xlabel('Galactic Longitude')
plt.ylabel('Galactic Latitude')

plt.subplot(335,projection=wcs[j1:j2,i1:i2])
plt.imshow(C_RM[j1:j2,i1:i2],vmin=-RMlim,vmax=RMlim,origin='lower',cmap='RdBu')
plt.contour(G_PI[j1:j2,i1:i2], levels=[PI_lim], colors='k', alpha=0.5)
plt.xlabel('Galactic Longitude')
plt.ylabel('Galactic Latitude')

plt.subplot(336,projection=wcs[j1:j2,i1:i2])
plt.imshow(CG_RM[j1:j2,i1:i2],vmin=-RMlim,vmax=RMlim,origin='lower',cmap='RdBu')
plt.contour(G_PI[j1:j2,i1:i2], levels=[PI_lim], colors='k', alpha=0.5)
plt.xlabel('Galactic Longitude')
plt.ylabel('Galactic Latitude')


plt.subplot(337,projection=wcs[j1:j2,i1:i2])
#plt.imshow(G_RM_good[j1:j2,i1:i2],vmin=-RMlim,vmax=RMlim,origin='lower',cmap='RdBu')
plt.contour(G_PI[j1:j2,i1:i2], levels=[PI_lim], colors='k', alpha=0.5)
plt.xlabel('Galactic Longitude')
plt.ylabel('Galactic Latitude')

plt.subplot(338,projection=wcs[j1:j2,i1:i2])
plt.imshow(C_RM_good[j1:j2,i1:i2],vmin=-RMlim,vmax=RMlim,origin='lower',cmap='RdBu')
plt.contour(G_PI[j1:j2,i1:i2], levels=[PI_lim], colors='k', alpha=0.5)
plt.xlabel('Galactic Longitude')
plt.ylabel('Galactic Latitude')

plt.subplot(339,projection=wcs[j1:j2,i1:i2])
plt.imshow(CG_RM_good[j1:j2,i1:i2],vmin=-RMlim,vmax=RMlim,origin='lower',cmap='RdBu')
plt.contour(G_PI[j1:j2,i1:i2], levels=[PI_lim], colors='k', alpha=0.5)
plt.xlabel('Galactic Longitude')
plt.ylabel('Galactic Latitude')

## Full CGPS plots for poster

In [ ]:
fs = 18
fig = plt.figure(figsize=(80, 20))
fig.subplots_adjust(left=0.05, right=0.99, top=0.99, bottom=0.05,hspace=0.001)

PIlim1 = 0.5
RMlim = 300
PI_lim = 0.1
fraction = 0.08
pad = 0.005
asp = 10

c = SkyCoord([192,52], [-3,5], frame=Galactic, unit="deg")
wcs = WCS(hdu_G_PI[0].header)
x, y = np.round(wcs.world_to_pixel(c))
i1 = int(x[0])
i2 = int(x[1])
j1 = int(y[0])
j2 = int(y[1])

plt.subplot(411,projection=wcs[j1:j2,i1:i2])
im = plt.imshow(CG_PI[j1:j2,i1:i2], vmin=0, vmax=PIlim1, origin='lower',cmap='gray')
plt.contour(G_PI[j1:j2,i1:i2], levels=[PI_lim], colors='red', alpha=0.5)
plt.ylabel('Galactic Latitude',fontsize=fs)
plt.xlabel('.')
plt.gca().tick_params(axis='x', labelsize=fs)
plt.gca().tick_params(axis='y', labelsize=fs)
cbar = plt.colorbar(im,pad=pad,fraction=fraction,aspect=asp,label='PI (K)')
cbar.ax.tick_params(labelsize=fs)
cbar.ax.yaxis.label.set_size(fs)

plt.subplot(412,projection=wcs[j1:j2,i1:i2])
im = plt.imshow(C_RM[j1:j2,i1:i2], vmin=-RMlim, vmax=RMlim, origin='lower',cmap='RdBu_r')
plt.contour(G_PI[j1:j2,i1:i2], levels=[PI_lim], colors='black', alpha=0.5)
plt.ylabel('Galactic Latitude',fontsize=fs)
plt.xlabel('.')
plt.gca().tick_params(axis='x', labelsize=fs)
plt.gca().tick_params(axis='y', labelsize=fs)
cbar = plt.colorbar(im,pad=pad,fraction=fraction,aspect=asp, label='RM (rad m$^{-2}$)')
cbar.ax.tick_params(labelsize=fs)
cbar.ax.yaxis.label.set_size(fs)

plt.subplot(413,projection=wcs[j1:j2,i1:i2])
im = plt.imshow(G_RM[j1:j2,i1:i2], vmin=-RMlim, vmax=RMlim, origin='lower',cmap='RdBu_r')
plt.contour(G_PI[j1:j2,i1:i2], levels=[PI_lim], colors='black', alpha=0.5)
plt.ylabel('Galactic Latitude',fontsize=fs)
plt.xlabel('.')
plt.gca().tick_params(axis='x', labelsize=fs)
plt.gca().tick_params(axis='y', labelsize=fs)
cbar = plt.colorbar(im,pad=pad,fraction=fraction,aspect=asp, label='RM (rad m$^{-2}$)')
cbar.ax.tick_params(labelsize=fs)
cbar.ax.yaxis.label.set_size(fs)

plt.subplot(414,projection=wcs[j1:j2,i1:i2])
im = plt.imshow(CG_RM[j1:j2,i1:i2], vmin=-RMlim, vmax=RMlim, origin='lower',cmap='RdBu_r')
plt.contour(G_PI[j1:j2,i1:i2], levels=[PI_lim], colors='black', alpha=0.5)
plt.ylabel('Galactic Latitude',fontsize=fs)
plt.xlabel('Galactic Longitude',fontsize=fs)
plt.gca().tick_params(axis='x', labelsize=fs)
plt.gca().tick_params(axis='y', labelsize=fs)
cbar = plt.colorbar(im,pad=pad,fraction=fraction,aspect=asp, label='RM (rad m$^{-2}$)')
cbar.ax.tick_params(labelsize=fs)
cbar.ax.yaxis.label.set_size(fs)

#plt.tight_layout()
plt.savefig('../CGPS_GMIMS_plots/poster_test.png')


probably best option is: combined PI, SA-RM, ST-RM, ST+SA-RM
maybe 1D RM vs long for both ST and SA+ST
(and all of these with convolved versions, not raw)

In [ ]:
fs = 18
fig = plt.figure(figsize=(80, 15))

PIlim1 = 0.5
RMlim = 300
PI_lim = 0.1
fraction = 0.08
pad = 0.005
asp = 10

c = SkyCoord([192,52], [-3,5], frame=Galactic, unit="deg")
wcs = WCS(hdu_G_PI[0].header)
x, y = np.round(wcs.world_to_pixel(c))
i1 = int(x[0])
i2 = int(x[1])
j1 = int(y[0])
j2 = int(y[1])

plt.subplot(411,projection=wcs[j1:j2,i1:i2])
im = plt.imshow(CG_PI[j1:j2,i1:i2], vmin=0, vmax=PIlim1, origin='lower',cmap='gray')
plt.contour(G_PI[j1:j2,i1:i2], levels=[PI_lim], colors='red', alpha=0.5)
plt.ylabel('Galactic Latitude',fontsize=fs)
#plt.yticks(fontsize=fs)
plt.gca().tick_params(axis='x', labelsize=fs)
plt.gca().tick_params(axis='y', labelsize=fs)
cbar = plt.colorbar(im,pad=pad,fraction=fraction,aspect=asp,label='PI (K)')
cbar.ax.tick_params(labelsize=fs)
cbar.ax.yaxis.label.set_size(fs)

plt.tight_layout()
#plt.savefig('../CGPS_GMIMS_plots/poster_test.png')

In [ ]:
print(wcs)

In [ ]:
print(ax.xaxis)

In [ ]:
data = np.random.rand(10, 10)

# Create an image plot of the data
fig = plt.subplots()
plt.subplot(411,projection=wcs[j1:j2,i1:i2])
im = plt.imshow(CG_PI[j1:j2,i1:i2], vmin=0, vmax=PIlim1, origin='lower')

# Add a colorbar to the plot
#divider = make_axes_locatable(plt.gca())
#cax = divider.append_axes("right", size="5%", pad=0.05)
cbar = plt.colorbar(im)
#cbar.ax.tick_params(labelsize=10)

In [ ]:
print(plt.gca()[0])

In [ ]:
fs = 24
fig = plt.figure(figsize=(20, 8))

PIlim = 0.4
RMlim = 400

c = SkyCoord([90,55], [-5,8], frame=Galactic, unit="deg")
wcs = WCS(hdu_G_PI[0].header)
x, y = np.round(wcs.world_to_pixel(c))
i1 = int(x[0])
i2 = int(x[1])
j1 = int(y[0])
j2 = int(y[1])

fraction = 0.018
pad = 0.005
asp = 20

plt.subplot(111,projection=wcs[j1:j2,i1:i2])
#im = plt.imshow(C_RM[j1:j2,i1:i2],vmin=-RMlim,vmax=RMlim,origin='lower',cmap='RdBu')
im = plt.imshow(CG_RM[j1:j2,i1:i2],vmin=-RMlim,vmax=RMlim,origin='lower',cmap='RdBu')
#plt.contour(G_PI[j1:j2,i1:i2], levels=[PI_lim], colors='k', alpha=0.5)
plt.xlabel('Galactic Longitude',fontsize=fs)
plt.ylabel('Galactic Latitude',fontsize=fs)
plt.tick_params(axis='both', which='major', labelsize=fs)

cbar = plt.colorbar(im,pad=pad,fraction=fraction,aspect=asp, label='RM (rad m$^{-2}$)')
cbar.ax.tick_params(labelsize=fs)
cbar.ax.yaxis.label.set_size(fs)

plt.savefig('/home/aordog/Python/cgps-gmims/CGPS_GMIMS_example_map.jpg')